# T07: Domain Quickstarts — All 13 Domains

## What You'll Learn

Spindle ships with **13 pre-built business domains**, each containing a complete
relational schema with realistic distributions and referential integrity. This
notebook generates every domain and gives you a side-by-side comparison:

1. **Generate** all 13 domains at demo scale
2. **Compare** table counts, total rows, and key tables for each
3. **Highlight** unique features per domain (GBM pricing, ICD-10 codes, etc.)
4. **Rank** domains by size and complexity

## The 13 Domains

| Domain | Focus |
|--------|-------|
| Retail | Customers, orders, products, stores |
| Healthcare | Patients, encounters, claims, ICD-10 |
| Financial | Accounts, transactions, loans |
| Supply Chain | Suppliers, purchase orders, shipments |
| IoT | Devices, telemetry, alerts |
| HR | Employees, departments, payroll |
| Insurance | Policies, claims, underwriting |
| Marketing | Campaigns, leads, conversions |
| Education | Students, courses, enrollments |
| Real Estate | Properties, listings, transactions |
| Manufacturing | Work orders, BOM, quality |
| Telecom | Subscribers, CDRs, network |
| Capital Markets | Companies, daily prices, trades |

## Prerequisites

- `sqllocks-spindle` installed

## Time Estimate

**~15 minutes**

In [1]:
# Uncomment the line below if sqllocks-spindle is not yet installed.
# %pip install sqllocks-spindle

print("Installation cell ready. Uncomment the pip line above if needed.")

Installation cell ready. Uncomment the pip line above if needed.


## Step 1 — Import All 13 Domains and Generate

**What we're about to do:** Import every domain class, generate each at `fabric_demo`
scale, and collect the results for comparison.

**Why this matters:** Seeing all domains generated in one place gives you a complete
picture of what Spindle offers. The `fabric_demo` scale is fast enough to generate
all 13 in under a minute.

In [2]:
import time
from sqllocks_spindle import (
    Spindle,
    RetailDomain,
    HealthcareDomain,
    FinancialDomain,
    SupplyChainDomain,
    IoTDomain,
    HrDomain,
    InsuranceDomain,
    MarketingDomain,
    EducationDomain,
    RealEstateDomain,
    ManufacturingDomain,
    TelecomDomain,
)
from sqllocks_spindle.domains.capital_markets import CapitalMarketsDomain

ALL_DOMAINS = [
    ("Retail",           RetailDomain()),
    ("Healthcare",       HealthcareDomain()),
    ("Financial",        FinancialDomain()),
    ("Supply Chain",     SupplyChainDomain()),
    ("IoT",              IoTDomain()),
    ("HR",               HrDomain()),
    ("Insurance",        InsuranceDomain()),
    ("Marketing",        MarketingDomain()),
    ("Education",        EducationDomain()),
    ("Real Estate",      RealEstateDomain()),
    ("Manufacturing",    ManufacturingDomain()),
    ("Telecom",          TelecomDomain()),
    ("Capital Markets",  CapitalMarketsDomain()),
]

results = {}
timings = {}
spindle = Spindle()

for label, domain in ALL_DOMAINS:
    t0 = time.perf_counter()
    data = spindle.generate(domain=domain, scale="fabric_demo", seed=42)
    elapsed = time.perf_counter() - t0
    results[label] = data
    timings[label] = elapsed

print(f"Generated all {len(ALL_DOMAINS)} domains successfully.")

Generated all 13 domains successfully.


## Step 2 — Compare Table Counts, Total Rows, and Key Tables

**What we're about to do:** Print a summary table showing each domain's table count,
total row count, generation time, and the names of its key tables.

**Why this matters:** This comparison lets you quickly identify which domain fits your
use case — and understand the relative complexity of each one.

In [3]:
print(f"{'Domain':<18} {'Tables':>6} {'Total Rows':>11} {'Time (s)':>9}   Key Tables")
print("-" * 95)

for label, data in results.items():
    tables = data.tables if hasattr(data, 'tables') else data
    table_count = len(tables)
    total_rows = sum(len(df) for df in tables.values())
    elapsed = timings[label]
    key_tables = ", ".join(list(tables.keys())[:4])
    if table_count > 4:
        key_tables += f", ... (+{table_count - 4})"
    print(
        f"  {label:<16} {table_count:>6} {total_rows:>11,} {elapsed:>8.2f}s   {key_tables}"
    )

grand_total = sum(
    sum(len(df) for df in (d.tables if hasattr(d, 'tables') else d).values())
    for d in results.values()
)
print(f"\n  {'TOTAL':<16} {'':>6} {grand_total:>11,}")

Domain             Tables  Total Rows  Time (s)   Key Tables
-----------------------------------------------------------------------------------------------
  Retail                9       4,670     0.21s   customer, address, product_category, product, ... (+5)
  Healthcare            9       4,282     0.04s   facility, patient, provider, encounter, ... (+5)
  Financial            10       3,596     0.03s   branch, customer, account, card, ... (+6)
  Supply Chain         10       5,322     0.11s   material, demand_forecast, supplier, purchase_order, ... (+6)
  IoT                   8       3,045     0.02s   device_type, location, device, alert, ... (+4)
  HR                    9       1,775     0.03s   department, position, employee, compensation, ... (+5)
  Insurance             9       2,255     0.03s   agent, policy_type, policyholder, policy, ... (+5)
  Marketing            10       4,440     0.33s   campaign_type, campaign, industry, lead_source, ... (+6)
  Education             9

## Step 3 — Unique Features per Domain

**What we're about to do:** Highlight what makes each domain special — the unique
data generation techniques, reference datasets, or column types that set it apart.

**Why this matters:** Knowing these features helps you pick the right domain and
understand what kind of analytical questions the generated data can answer.

In [4]:
domain_features = {
    "Retail": "Loyalty tiers, product categories, seasonal order patterns, store regions",
    "Healthcare": "ICD-10 diagnosis codes (CMS frequencies), CPT procedures, payer mix, claim denial simulation",
    "Financial": "Account types, transaction categories, loan amortization schedules, credit scoring",
    "Supply Chain": "Multi-tier suppliers, purchase orders, shipment tracking, warehouse inventory",
    "IoT": "Device telemetry with time-series patterns, alert thresholds, sensor drift simulation",
    "HR": "Org hierarchy, salary bands by role/level, performance reviews, turnover modeling",
    "Insurance": "Policy lifecycle, underwriting risk scores, claim severity distributions, actuarial tables",
    "Marketing": "Campaign funnels, lead scoring, A/B test cohorts, conversion attribution",
    "Education": "Course catalogs, enrollment patterns, GPA distributions, graduation tracking",
    "Real Estate": "Property valuations, listing lifecycle, agent performance, market comparables",
    "Manufacturing": "Bill of materials (BOM), work order routing, quality inspections, OEE metrics",
    "Telecom": "Subscriber plans, call detail records (CDRs), network utilization, churn indicators",
    "Capital Markets": "Real S&P 500 tickers, GBM daily price simulation, EPS surprise, insider trades (SEC Form 4)",
}

print("Unique Features by Domain")
print("=" * 70)
for label, features in domain_features.items():
    tables = results[label].tables if hasattr(results[label], 'tables') else results[label]
    print(f"\n  {label} ({len(tables)} tables)")
    print(f"    {features}")

Unique Features by Domain

  Retail (9 tables)
    Loyalty tiers, product categories, seasonal order patterns, store regions

  Healthcare (9 tables)
    ICD-10 diagnosis codes (CMS frequencies), CPT procedures, payer mix, claim denial simulation

  Financial (10 tables)
    Account types, transaction categories, loan amortization schedules, credit scoring

  Supply Chain (10 tables)
    Multi-tier suppliers, purchase orders, shipment tracking, warehouse inventory

  IoT (8 tables)
    Device telemetry with time-series patterns, alert thresholds, sensor drift simulation

  HR (9 tables)
    Org hierarchy, salary bands by role/level, performance reviews, turnover modeling

  Insurance (9 tables)
    Policy lifecycle, underwriting risk scores, claim severity distributions, actuarial tables

  Marketing (10 tables)
    Campaign funnels, lead scoring, A/B test cohorts, conversion attribution

  Education (9 tables)
    Course catalogs, enrollment patterns, GPA distributions, graduation tra

## Step 4 — Rank Domains by Size and Complexity

**What we're about to do:** Sort domains by total row count and table count to
visualize the relative size and complexity of each.

**Why this matters:** Larger domains take longer to generate at higher scales.
Understanding the baseline helps you choose appropriate scales for your workload
and estimate generation times for production-sized datasets.

In [5]:
import pandas as pd

rows_data = []
for label, data in results.items():
    tables = data.tables if hasattr(data, 'tables') else data
    total_rows = sum(len(df) for df in tables.values())
    rows_data.append({
        "Domain": label,
        "Tables": len(tables),
        "Total Rows": total_rows,
        "Gen Time (s)": round(timings[label], 2),
    })

ranking = pd.DataFrame(rows_data).sort_values("Total Rows", ascending=False).reset_index(drop=True)
ranking.index = ranking.index + 1  # 1-based ranking
ranking.index.name = "Rank"

print("Domains Ranked by Total Rows (fabric_demo scale)")
print("=" * 60)
print(ranking.to_string())

print(f"\nSmallest: {ranking.iloc[-1]['Domain']} ({ranking.iloc[-1]['Total Rows']:,} rows)")
print(f"Largest:  {ranking.iloc[0]['Domain']} ({ranking.iloc[0]['Total Rows']:,} rows)")

Domains Ranked by Total Rows (fabric_demo scale)
               Domain  Tables  Total Rows  Gen Time (s)
Rank                                                   
1     Capital Markets      10      126130          0.07
2             Telecom       9       14740          0.11
3        Supply Chain      10        5322          0.11
4              Retail       9        4670          0.21
5           Marketing      10        4440          0.33
6          Healthcare       9        4282          0.04
7           Financial      10        3596          0.03
8           Education       9        3447          0.05
9                 IoT       8        3045          0.02
10          Insurance       9        2255          0.03
11                 HR       9        1775          0.03
12        Real Estate       9        1555          0.04
13      Manufacturing       9        1515          0.01

Smallest: Manufacturing (1,515 rows)
Largest:  Capital Markets (126,130 rows)


## What's Next?

You've seen all 13 domains at a glance. Pick the one that matches your use case and
dive deeper:

| Notebook | What You'll Learn |
|----------|-------------------|
| **T04: Healthcare Deep Dive** | Explore healthcare tables, ICD-10 distributions, clinical flows |
| **T09: Capital Markets Deep Dive** | S&P 500 tickers, GBM pricing, insider trades |
| **F07: Healthcare RCM** | Build an RCM analytics pipeline from synthetic data |
| **T05: Distribution Overrides** | Customize any domain's distributions |

**Key takeaways:**
- All 13 domains generate at demo scale in under a minute combined
- Each domain has unique, domain-specific data generation techniques
- Table counts range from simple (5-6 tables) to complex (10+ tables)
- Every domain supports the same API: `Spindle.generate(domain=..., scale=..., seed=...)`